In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gymnasium as gym
from scipy import stats

# Hyperparameters
H, BATCH, GAMMA, TAU, LR = 128, 128, 0.99, 0.005, 3e-4


# Actor Network
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim, max_a):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(s_dim, H),
            nn.ReLU(),
            nn.Linear(H, H),
            nn.ReLU()
        )

        self.mean = nn.Linear(H, a_dim)
        self.log_std = nn.Linear(H, a_dim)
        self.max_a = max_a

    def sample(self, s):
        x = self.net(s)

        mean = self.mean(x)
        log_std = self.log_std(x).clamp(-20, 2)

        dist = torch.distributions.Normal(
            mean, log_std.exp()
        )

        z = dist.rsample()

        action = torch.tanh(z) * self.max_a

        log_prob = (
            dist.log_prob(z)
            - torch.log(
                1 - torch.tanh(z).pow(2) + 1e-6
            )
        ).sum(-1, keepdim=True)

        return action, log_prob


# Critic Network
class Critic(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(s_dim + a_dim, H),
            nn.ReLU(),
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Linear(H, 1)
        )

    def forward(self, s, a):
        return self.net(torch.cat([s, a], -1))


# Soft Actor-Critic
class SAC:
    def __init__(self, s_dim, a_dim, max_a):

        self.actor = Actor(s_dim, a_dim, max_a)

        self.q1 = Critic(s_dim, a_dim)
        self.q2 = Critic(s_dim, a_dim)

        self.tq1 = Critic(s_dim, a_dim)
        self.tq2 = Critic(s_dim, a_dim)

        self.tq1.load_state_dict(
            self.q1.state_dict()
        )

        self.tq2.load_state_dict(
            self.q2.state_dict()
        )

        self.opt_a = optim.Adam(
            self.actor.parameters(),
            lr=LR
        )

        self.opt_q1 = optim.Adam(
            self.q1.parameters(),
            lr=LR
        )

        self.opt_q2 = optim.Adam(
            self.q2.parameters(),
            lr=LR
        )

        self.log_alpha = torch.zeros(
            1,
            requires_grad=True
        )

        self.opt_alpha = optim.Adam(
            [self.log_alpha],
            lr=LR
        )

        self.target_entropy = -a_dim

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def act(self, s):

        with torch.no_grad():

            a, _ = self.actor.sample(
                torch.FloatTensor(s).unsqueeze(0)
            )

        return a.squeeze(0).numpy()

    def update(self, buf):

        if len(buf) < BATCH:
            return

        s, a, r, s2, d = buf.sample(BATCH)

        # Calculate target Q-value
        with torch.no_grad():

            a2, logp2 = self.actor.sample(s2)

            q_target = (
                torch.min(
                    self.tq1(s2, a2),
                    self.tq2(s2, a2)
                )
                - self.alpha * logp2
            )

            y = r + (1 - d) * GAMMA * q_target

        # Train critics
        for q, opt in [
            (self.q1, self.opt_q1),
            (self.q2, self.opt_q2)
        ]:

            loss = nn.MSELoss()(q(s, a), y)

            opt.zero_grad()
            loss.backward()
            opt.step()

        # Train actor
        new_a, logp = self.actor.sample(s)

        q_min = torch.min(
            self.q1(s, new_a),
            self.q2(s, new_a)
        )

        actor_loss = (
            self.alpha * logp - q_min
        ).mean()

        self.opt_a.zero_grad()
        actor_loss.backward()
        self.opt_a.step()

        # Automatic alpha tuning
        alpha_loss = -(
            self.log_alpha
            * (logp + self.target_entropy).detach()
        ).mean()

        self.opt_alpha.zero_grad()
        alpha_loss.backward()
        self.opt_alpha.step()

        # Soft update target critics
        for net, tnet in [
            (self.q1, self.tq1),
            (self.q2, self.tq2)
        ]:

            for p, tp in zip(
                net.parameters(),
                tnet.parameters()
            ):

                tp.data.copy_(
                    TAU * p.data
                    + (1 - TAU) * tp.data
                )


# Replay Buffer
class Buffer:

    def __init__(self, cap=100_000):
        self.d = deque(maxlen=cap)

    def push(self, *x):
        self.d.append(x)

    def sample(self, n):

        s, a, r, s2, d = map(
            np.array,
            zip(*random.sample(self.d, n))
        )

        return (
            torch.FloatTensor(s),
            torch.FloatTensor(a),
            torch.FloatTensor(r).unsqueeze(1),
            torch.FloatTensor(s2),
            torch.FloatTensor(d).unsqueeze(1)
        )

    def __len__(self):
        return len(self.d)


# Create environment
env = gym.make("Pendulum-v1")

agent = SAC(
    env.observation_space.shape[0],
    env.action_space.shape[0],
    float(env.action_space.high[0])
)

buf = Buffer()

reward_history = []


# Training
for ep in range(50):

    s, _ = env.reset()

    done = False
    ep_r = 0

    while not done:

        if len(buf) < 500:
            a = env.action_space.sample()
        else:
            a = agent.act(s)

        s2, r, term, trunc, _ = env.step(a)

        done = term or trunc

        buf.push(
            s,
            a,
            r,
            s2,
            float(done)
        )

        s = s2

        ep_r += r

        agent.update(buf)

    reward_history.append(ep_r)

    print(
        f"Episode {ep + 1}: "
        f"reward={ep_r:.1f}, "
        f"alpha={agent.alpha.item():.3f}"
    )


# Evaluation
last_n = reward_history[-10:]

mean_reward = np.mean(last_n)
sem_reward = stats.sem(last_n)

print(
    f"\nFinal performance "
    f"(last {len(last_n)} episodes): "
    f"{mean_reward:.1f} +/- {sem_reward:.1f} "
    f"(mean +/- SEM)"
)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\mahab\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.